In [ ]:
import numpy as np
import torch

In [ ]:
X = torch.randn((32, 8, 2))
Y = torch.randn((32, ))

In [ ]:
torch.manual_seed(42)

In [ ]:
class Linear():
  def __init__(self, input_dim, output_dim, set_bias=False):
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.weights = torch.randn((input_dim, output_dim))
    self.bias = torch.ones((output_dim)) if set_bias else None

  def __call__(self, x):
    self.out = x @ self.weights + self.bias if self.bias is not None else x @ self.weights
    return self.out

  def __repr__(self):
    return f"Linear(input_dim={self.input_dim}, output_dim={self.output_dim})"

  def parameters(self):
    return [self.weights] + ([] if self.bias is None else [self.bias])


class Tanh():
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out

  def __repr__(self):
    return f"Tanh()"

  def parameters(self):
    return []


class ConsecutiveFlatten():
  def __init__(self, n):
    self.n = n

  def __repr__(self):
    return f"ConsecutiveFlatten(n={self.n})"

  def __call__(self, x):

    B, R, C = x.shape
    flattened = x.view(B, R//self.n, C*self.n)

    if flattened.shape[1] == 1:
      flattened = flattened.squeeze(1)


    return flattened

  def parameters(self):
    return []



class Sequential():
  def __init__(self, layers):
    self.layers = layers
    self.params = []

  def __repr__(self):
    s = []
    for layer in self.layers:
      s.append(layer)
    return f"Sequential(\n{s}\n)"

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    self.out = x
    return x

  def parameters(self):
    self.params = [p for layer in self.layers for p in layer.parameters()]
    return self.params


In [ ]:
class BasicRNN():
  def __init__(self, n_features, n_neurons, state=None):
    self.hidden = n_neurons
    self.input = n_features
    self.W_x = torch.randn((n_features, n_neurons))
    self.W_h = torch.randn((n_neurons, n_neurons))
    self.b_h = torch.zeros((n_neurons))
    self.state = state

  def __call__(self, x):
    if self.state is None:
      self.state = torch.zeros((x.shape[1], self.hidden))

    output = []
    for input in x:
      self.state = torch.tanh(
          input @ self.W_x +
          self.state @ self.W_h +
          self.b_h
      )
      output.append(self.state)

    return output, self.state

  def __repr__(self):
    return f"BasicRNN(n_inputs={self.input}, n_output={self.hidden})"

  def parameters(self):
    return [self.W_x, self.W_h, self.b_h]








In [ ]:
X = torch.randn((10, 32, 4))
Y = torch.randn((10, 32))

In [ ]:
hidden = 20
rnn = BasicRNN(X.shape[-1], hidden)
linear = Linear(hidden, 10, True)
rnn

BasicRNN(n_inputs=4, n_output=20)

In [ ]:
import torch.nn.functional as F

model_params = rnn.parameters() + linear.parameters()
for p in model_params:
  p.requires_grad = True

epochs = 10
for i in range(epochs):
  #forward pass
  outputs, state = rnn(X)
  logits = linear(state)
  print(logits.shape)
  print(Y.shape)
  #logits = logits.squeeze(-1)
  loss = F.mse_loss(logits, Y)


  #backward pass
  for p in model_params:
    p.grad = None

  loss.backward(retain_graph=True)

  lr = 0.01
  for p in model_params:
    p.data += -lr * p.grad
  break
print(loss.item())



torch.Size([32, 10])
torch.Size([10, 32])


/tmp/ipykernel_17883/546014642.py:15: UserWarning: Using a target size (torch.Size([10, 32])) that is different to the input size (torch.Size([32, 10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(logits, Y)


RuntimeError: The size of tensor a (10) must match the size of tensor b (32) at non-singleton dimension 1

In [ ]:
n = 2
n_features = X.shape[-1]
n_hidden = 20
layers = [ConsecutiveFlatten(n), Linear(n_features*n, n_hidden), Tanh(),
          ConsecutiveFlatten(n), Linear(n_hidden*n, n_hidden), Tanh(),
          ConsecutiveFlatten(n), Linear(n_hidden*n, n_hidden), Tanh(),
          Linear(n_hidden, 1)
          ]
model = Sequential(layers)
model.layers[-1].weights *= 0.1

for p in model.parameters():
  p.requires_grad = True

In [ ]:

epochs = 100
for i in range(epochs):
  #forward pass
  logits = model(X).squeeze(1)
  loss = F.mse_loss(logits, Y)


  #backward pass
  for p in model.parameters():
    p.grad = None

  loss.backward()

  lr = 0.1
  for p in model.parameters():
    p.data += -lr * p.grad

print(loss.item())




0.0026888353750109673


In [ ]:
X = torch.randn((32, 4))
Y = torch.randn((32, ))

In [ ]:
n_features = 4
n_hidden = 20
mlp_layers = [
    Linear(n_features, n_hidden), Tanh(),
    Linear(n_hidden, n_hidden//2), Tanh(),
    Linear(n_hidden//2, 1),
]

mlp = Sequential(mlp_layers)
mlp.layers[-1].weights *= 0.1

for p in mlp.parameters():
  p.requires_grad = True

mlp

Sequential(
[Linear(input_dim=4, output_dim=20), Tanh(), Linear(input_dim=20, output_dim=10), Tanh(), Linear(input_dim=10, output_dim=1)]
)

In [ ]:
for i in range(epochs):
  logits = mlp(X)
  logits = logits.squeeze(1)
  loss = F.mse_loss(logits, Y)

  for p in mlp.parameters():
    p.grad = None
  loss.backward()

  lr = 0.35
  for p in mlp.parameters():
    p.data += -lr * p.grad


loss.item()

0.010077370330691338